In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l0.5 Break generation

> 🎯 **Goal:** deliberately make the model fail by looping, understand exactly why, and connect the fix back to sampling.

**Why this matters.** You learn a system best by breaking it on purpose. The repetition loop you are about to trigger is not a quirk of our toy model; it is a real failure mode that shows up in production language models too, and "why does it keep repeating itself" is one of the most common questions practitioners ask. Seeing it happen on a model you control, and knowing the cure, means you will recognize and fix it anywhere.

**The intuition.** Greedy decoding (the `temperature=0` you met last lesson) always grabs the single most likely next character. That sounds safe, but imagine a path that leads back to where it started: if "and t" most-likely leads to "h", and "and th" most-likely leads to "e", and "and the" most-likely leads back to " and t", the model walks in a circle forever. With no randomness to nudge it off the track, it cannot escape. A tiny, barely-trained model has lots of these little ruts.

**The mechanics.** We generate 200 characters with `temperature=0` and watch it collapse into a cycle. Then we *measure* the collapse: a healthy 200-character sample uses many different characters fairly evenly, but a looped one is dominated by a handful that recur over and over. We use Python's `Counter` to tally how often each character appears; a high top count is the fingerprint of repetition. The cure you already know: any form of sampling (temperature above 0, top_k, or top_p) gives lower-probability characters a chance and breaks the cycle.

In [ ]:
from collections import Counter
from pocketlm import train_tiny, pick_config, generate

corpus = open(CORPUS_PATH, encoding="utf-8").read()
model, tok = train_tiny(corpus, pick_config(DEVICE, 1), seed=0)
broken = generate(model, tok, "The ", max_new_tokens=200, temperature=0)
print(broken)

**What just happened.** The output above almost certainly fell into a loop, a short sequence of characters repeating with little new information. That is greedy decoding eating itself: every step it picks the locally safest character, and on a small model those safe choices chain into a cycle it cannot leave. This is the failure l0.4 hinted at, now staring back at you.

Sampling is the escape hatch. Temperature above 0, `top_k`, or `top_p` all inject just enough chance for a lower-probability character to win occasionally, knocking the model out of the rut and back onto fresh text. The loop also leaves a measurable signature: a few characters dominate the output. The cell below counts characters to prove it.

In [ ]:
counts = Counter(broken)
print("most common:", counts.most_common(5))
assert max(counts.values()) >= 3  # the same characters recur

**What just happened.** The `most common` list shows a small handful of characters accounting for a big share of the 200 we generated, and the assert confirms the top character recurs at least three times. That lopsided tally is the quantitative fingerprint of the loop you saw above: when generation is healthy and varied, no single character runs away with the count like this.

⚠️ **Common pitfalls**
- Blaming the model's *training* for repetition. Often it is the *decoding*. The same weights, decoded with sampling, can produce perfectly varied text. Always check your decoding settings before assuming the model is broken.
- Assuming bigger models never loop. They loop less, but greedy decoding can still trap them, which is why real systems rarely decode at pure temperature 0 for long generations.

🏋️ **Try it yourself.** Re-run the generation cell but change `temperature=0` to `temperature=0.8` (or add `top_k=10`), then re-run this counting cell. Watch the most-common counts flatten out as sampling breaks the loop. That contrast, broken versus healthy, is the whole point of this lesson, and a fitting close to unit u0: you have now seen a language model speak, tokenize, assign probabilities, decode, and fail, the full life cycle of generation before we open up how it all works inside.